# SAM2-GHAN Dataset — Clean Zip Builder
Extracts only the needed files from the previous notebook output and creates a clean dataset zip.

**Needed:**
```
SAM2_GHAN_Dataset_Ready/
├── train/images/ + train/masks/
├── val/  images/ + val/  masks/
├── test/ images/ + test/ masks/
└── metadata_hierarchy.csv
class_split_info.json
label_mappings.json
```

**Skipped:** `.virtual_documents`, `sam2/`, `segmented_raw/`

In [1]:
import os, zipfile, shutil
from pathlib import Path
from tqdm.notebook import tqdm

# ── CONFIG ────────────────────────────────────────────────────────────────────
SRC_ZIP   = "/kaggle/input/notebooks/zahidhasantonmoy/sam2-ghan-dataset-builder/_output_.zip"
WORK_DIR  = "/kaggle/working/extracted"
OUT_ZIP   = "/kaggle/working/SAM2_GHAN_Clean_Dataset.zip"

# Only these top-level entries will be kept
KEEP = {
    "SAM2_GHAN_Dataset_Ready",
    "class_split_info.json",
    "label_mappings.json",
}
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(WORK_DIR, exist_ok=True)

# ── Step 1: Inspect zip contents ──────────────────────────────────────────────
print("Reading zip file...")
with zipfile.ZipFile(SRC_ZIP, 'r') as z:
    all_entries = z.namelist()

print(f"Total entries in zip: {len(all_entries):,}")

# Show top-level items
top_level = sorted(set(
    Path(e).parts[0] for e in all_entries if e.strip('/')
))
print("\nTop-level items:")
for t in top_level:
    count = sum(1 for e in all_entries if Path(e).parts and Path(e).parts[0] == t)
    print(f"  {t}/  ({count:,} entries)")

# Filter only needed entries
needed = [
    e for e in all_entries
    if Path(e).parts and Path(e).parts[0] in KEEP
]
print(f"\nEntries to keep: {len(needed):,}")
print(f"Entries to skip: {len(all_entries) - len(needed):,}")

Reading zip file...
Total entries in zip: 266,209

Top-level items:
  SAM2_GHAN_Dataset_Ready/  (195,304 entries)
  SAM2_GHAN_Dataset_Ready.zip/  (1 entries)
  __pycache__/  (1 entries)
  class_split_info.json/  (1 entries)
  sam2/  (733 entries)
  segmented_raw/  (70,169 entries)

Entries to keep: 195,305
Entries to skip: 70,904


In [2]:
import os, zipfile
from pathlib import Path
from tqdm.notebook import tqdm

SRC_ZIP  = "/kaggle/input/notebooks/zahidhasantonmoy/sam2-ghan-dataset-builder/_output_.zip"
OUT_ZIP  = "/kaggle/working/SAM2_GHAN_Clean_Dataset.zip"
KEEP     = {"SAM2_GHAN_Dataset_Ready", "class_split_info.json", "label_mappings.json"}

# ── Step 2: Stream directly src → dst zip (no full extract needed) ─────────────
# This avoids writing 10GB to disk — reads src and writes only needed entries.
print("Building clean zip (streaming, no full extract)...")
print("This may take 10–20 minutes for ~10 GB source.\n")

written   = 0
skipped   = 0
bytes_out = 0

with zipfile.ZipFile(SRC_ZIP, 'r') as src_z, \
     zipfile.ZipFile(OUT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED,
                     compresslevel=1) as dst_z:

    all_entries = src_z.namelist()
    needed = [
        e for e in all_entries
        if Path(e).parts and Path(e).parts[0] in KEEP
    ]

    pbar = tqdm(needed, desc="Copying entries")
    for entry in pbar:
        info = src_z.getinfo(entry)

        # Skip directory entries (folders) — they get created automatically
        if info.filename.endswith('/'):
            skipped += 1
            continue

        # Read from source and write to destination
        data = src_z.read(entry)
        dst_z.writestr(info, data)
        written   += 1
        bytes_out += len(data)

        del data  # free RAM immediately

        if written % 5000 == 0:
            pbar.set_postfix(
                files=written,
                out_gb=f"{bytes_out/1e9:.2f}GB"
            )

print(f"\n=== DONE ===")
print(f"Files written : {written:,}")
print(f"Dirs skipped  : {skipped:,}")

out_size = os.path.getsize(OUT_ZIP) / (1024**3)
print(f"Output zip    : SAM2_GHAN_Clean_Dataset.zip")
print(f"Output size   : {out_size:.2f} GB")

Building clean zip (streaming, no full extract)...
This may take 10–20 minutes for ~10 GB source.



Copying entries:   0%|          | 0/195305 [00:00<?, ?it/s]


=== DONE ===
Files written : 193,945
Dirs skipped  : 1,360
Output zip    : SAM2_GHAN_Clean_Dataset.zip
Output size   : 3.58 GB


In [3]:
import zipfile
from pathlib import Path
from collections import defaultdict

OUT_ZIP = "/kaggle/working/SAM2_GHAN_Clean_Dataset.zip"

# ── Step 3: Verify clean zip contents ─────────────────────────────────────────
print("Verifying clean zip...\n")

with zipfile.ZipFile(OUT_ZIP, 'r') as z:
    entries = [e for e in z.namelist() if not e.endswith('/')]

# Count by split and type
split_img  = defaultdict(int)
split_mask = defaultdict(int)
csvs = []
jsons = []

for e in entries:
    parts = Path(e).parts  # e.g. ('SAM2_GHAN_Dataset_Ready','train','images','cobra','img.jpg')
    if len(parts) >= 3 and parts[0] == 'SAM2_GHAN_Dataset_Ready':
        split = parts[1]  # train / val / test
        kind  = parts[2]  # images / masks
        if kind == 'images': split_img[split]  += 1
        if kind == 'masks':  split_mask[split] += 1
    elif e.endswith('.csv'):  csvs.append(e)
    elif e.endswith('.json'): jsons.append(e)

print(f"{'Split':<8} {'Images':>8} {'Masks':>8} {'Paired?':>8}")
print('-' * 36)
for split in ['train', 'val', 'test']:
    imgs  = split_img.get(split, 0)
    masks = split_mask.get(split, 0)
    ok    = 'YES' if imgs == masks else f'NO ({imgs - masks} diff)'
    print(f"{split:<8} {imgs:>8,} {masks:>8,} {ok:>8}")

total_imgs  = sum(split_img.values())
total_masks = sum(split_mask.values())
print('-' * 36)
print(f"{'TOTAL':<8} {total_imgs:>8,} {total_masks:>8,}")

print(f"\nCSV  files : {csvs}")
print(f"JSON files : {jsons}")

import os
gb = os.path.getsize(OUT_ZIP) / (1024**3)
print(f"\nFinal zip size: {gb:.2f} GB")
print("\nDataset is clean and ready to publish as a Kaggle dataset!")

Verifying clean zip...

Split      Images    Masks  Paired?
------------------------------------
train      90,000   90,000      YES
val         3,491    3,491      YES
test        3,480    3,480      YES
------------------------------------
TOTAL      96,971   96,971

CSV  files : ['SAM2_GHAN_Dataset_Ready/metadata_hierarchy.csv']
JSON files : ['class_split_info.json', 'SAM2_GHAN_Dataset_Ready/label_mappings.json']

Final zip size: 3.58 GB

Dataset is clean and ready to publish as a Kaggle dataset!


## Step 4 — Delete Unnecessary Files from Working Directory
> Frees up Kaggle disk space after clean zip is verified.

In [4]:
import os, shutil
from pathlib import Path

WORK = "/kaggle/working"

# ── What to DELETE ─────────────────────────────────────────────────────────────
TO_DELETE = [
    "/kaggle/working/extracted",          # temp extraction folder (if used)
    "/kaggle/working/segmented_raw",      # raw segmented images+masks (huge)
    "/kaggle/working/sam2_ghan_dataset",              # intermediate (if exists)
    "/kaggle/working/sam2_ghan_dataset_augmented",    # intermediate (if exists)
    "/kaggle/working/SAM2_GHAN_Dataset_Ready",        # already zipped, not needed
    "/kaggle/working/class_split_info.json",
    "/kaggle/working/split_records.json",
    "/kaggle/working/sam2",               # SAM2 source code clone
]

# ── What to KEEP ───────────────────────────────────────────────────────────────
# /kaggle/working/SAM2_GHAN_Clean_Dataset.zip  ← our final output

print("=== CLEANUP ===\n")

freed_bytes = 0

for path_str in TO_DELETE:
    p = Path(path_str)
    if not p.exists():
        print(f"  SKIP (not found) : {path_str}")
        continue

    # Calculate size before deleting
    if p.is_dir():
        size = sum(f.stat().st_size for f in p.rglob('*') if f.is_file())
        shutil.rmtree(p)
        print(f"  DELETED (dir)    : {path_str}  ({size/1e9:.2f} GB)")
    else:
        size = p.stat().st_size
        p.unlink()
        print(f"  DELETED (file)   : {path_str}  ({size/1e6:.1f} MB)")

    freed_bytes += size

print(f"\nTotal space freed: {freed_bytes/1e9:.2f} GB")

# ── Show what remains ──────────────────────────────────────────────────────────
print("\n=== WORKING DIRECTORY NOW ===")
for item in sorted(os.listdir(WORK)):
    full = os.path.join(WORK, item)
    if os.path.isdir(full):
        size = sum(f.stat().st_size for f in Path(full).rglob('*') if f.is_file())
        print(f"  {item}/  ({size/1e9:.2f} GB)")
    else:
        size = os.path.getsize(full)
        print(f"  {item}  ({size/1e9:.2f} GB)")


=== CLEANUP ===

  DELETED (dir)    : /kaggle/working/extracted  (0.00 GB)
  SKIP (not found) : /kaggle/working/segmented_raw
  SKIP (not found) : /kaggle/working/sam2_ghan_dataset
  SKIP (not found) : /kaggle/working/sam2_ghan_dataset_augmented
  SKIP (not found) : /kaggle/working/SAM2_GHAN_Dataset_Ready
  SKIP (not found) : /kaggle/working/class_split_info.json
  SKIP (not found) : /kaggle/working/split_records.json
  SKIP (not found) : /kaggle/working/sam2

Total space freed: 0.00 GB

=== WORKING DIRECTORY NOW ===
  SAM2_GHAN_Clean_Dataset.zip  (3.84 GB)
  __notebook__.ipynb  (0.00 GB)
